<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/2_BiLSTM_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv('/content/BBCA_Master_Dataset_BiLSTM.csv')

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date')


print(f"Data loaded successfully. Total rows: {len(df)}")
print("\n--- Data Verification ---")
print(df[['Date', 'Close', 'Sentiment_Score']].head(3))

# Drop any NaNs
df = df.dropna().reset_index(drop=True)

In [ ]:
# Define Features
features_base = ['Open', 'High', 'Low', 'Close', 'Volume']
features_enh = ['Open', 'High', 'Low', 'Close', 'Volume', 'Sentiment_Score']
target = ['Close']

# Initialize Scalers
scaler_features = MinMaxScaler()
scaler_target = MinMaxScaler()

# Scale based on the enhanced set (which includes everything)
scaled_all = scaler_features.fit_transform(df[features_enh])
scaled_target = scaler_target.fit_transform(df[target])

# SAVE SCALERS
joblib.dump(scaler_features, 'scaler_features.pkl')
joblib.dump(scaler_target, 'scaler_target.pkl')

def create_sequences(data, target_data, window_size=10):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size)])
        y.append(target_data[i + window_size])
    return np.array(X), np.array(y)

# Create sequences
WINDOW_SIZE = 10
X_base, y = create_sequences(scaled_all[:, :5], scaled_target, WINDOW_SIZE)
X_enh, _ = create_sequences(scaled_all, scaled_target, WINDOW_SIZE)

# Split 80/20 (Time-series split, not random)
split = int(len(X_base) * 0.8)
X_train_base, X_test_base = X_base[:split], X_base[split:]
X_train_enh, X_test_enh = X_enh[:split], X_enh[split:]
y_train, y_test = y[:split], y[split:]

print(f"Sequences created. Training on {len(X_train_base)} days, Testing on {len(X_test_base)} days.")

In [ ]:
def build_bilstm(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        # Bidirectional layer processes sequences in both directions
        Bidirectional(LSTM(units=64, return_sequences=True)),
        Dropout(0.2),
        # Second LSTM layer to condense information
        LSTM(units=32, return_sequences=False),
        Dropout(0.2),
        Dense(units=16, activation='relu'),
        Dense(units=1) # Predicting next day price
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Early stopping prevents 'overfitting' (memorization)
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

In [ ]:
# Train Baseline (Numerical Only)
print("Training Baseline Model...")
model_base = build_bilstm((WINDOW_SIZE, 5))
model_base.fit(X_train_base, y_train, validation_data=(X_test_base, y_test),
               epochs=100, batch_size=16, callbacks=[early_stop], verbose=0)

# Train Sentiment-Enhanced (With Lagged Sentiment)
print("Training Sentiment-Enhanced Model...")
model_enh = build_bilstm((WINDOW_SIZE, 6))
model_enh.fit(X_train_enh, y_train, validation_data=(X_test_enh, y_test),
              epochs=100, batch_size=16, callbacks=[early_stop], verbose=0)

# Save the final model
model_enh.save('BBCA_BiLSTM_Sentiment_Model.keras')
print("Models trained and Sentiment model saved.")

In [ ]:
# 1. Get Predictions
pred_base = model_base.predict(X_test_base)
pred_enh = model_enh.predict(X_test_enh)

# 2. Inverse Scale to Rupiah
actual_prices = scaler_target.inverse_transform(y_test)
prices_base = scaler_target.inverse_transform(pred_base)
prices_enh = scaler_target.inverse_transform(pred_enh)

# 3. Calculate Error (RMSE)
rmse_base = math.sqrt(mean_squared_error(actual_prices, prices_base))
rmse_enh = math.sqrt(mean_squared_error(actual_prices, prices_enh))

print(f"\n--- FINAL EVALUATION ---")
print(f"Baseline (Price Only) RMSE: Rp {rmse_base:.2f}")
print(f"Enhanced (Price + Sentiment) RMSE: Rp {rmse_enh:.2f}")
print(f"Accuracy Improvement: {((rmse_base - rmse_enh) / rmse_base) * 100:.2f}%")

# 4. Plotting
plt.figure(figsize=(15, 8))
plt.plot(actual_prices, label='Actual BBCA Price', color='black', linewidth=2)
plt.plot(prices_base, label=f'Baseline BiLSTM (RMSE: {rmse_base:.0f})', color='red', linestyle='--')
plt.plot(prices_enh, label=f'Sentiment BiLSTM (RMSE: {rmse_enh:.0f})', color='green', linestyle='-.')

plt.title('Thesis Result: Impact of Lagged News Sentiment on BBCA Prediction', fontsize=14)
plt.xlabel('Test Set Days', fontsize=12)
plt.ylabel('Price (IDR)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()